# 06B - Tokenization and Embeddings PyTorch Lab

**Guiding question:** can you trace a legal phrase from text to GPT-2 IDs, train only the embedding rows that participated, and prove padding contributes zero loss?

> **What you finished last time:** 06A established BPE, token-address, static/contextual, and padding contracts.
> **What this notebook delivers:** compact BPE helpers, real GPT-2 `tiktoken` inspection, a controlled `nn.Embedding` learning experiment, and a PyTorch masking proof.
> **What this chapter is not:** a contextual encoder or retrieval application.


In [ ]:
# -- Setup ------------------------------------------------------------------------
from collections import Counter
import re
import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt
import tiktoken

torch.manual_seed(42)
np.random.seed(42)
plt.rcParams.update({"figure.facecolor": "#1a1a2e", "axes.facecolor": "#1a1a2e"})

LEGAL_CORPUS = [
    "the non-disclosure agreement protects confidential information",
    "the contract protects confidential records",
    "the agreement lists payment obligations",
    "a non-disclosure clause limits disclosure",
    "the contract and agreement bind the company",
]
print(f"Corpus sentences: {len(LEGAL_CORPUS)}")


## 1 - A compact BPE you can read in one minute

```mermaid
flowchart LR
    A["word characters"] --> B["pair counts"]
    B --> C["best pair"]
    C --> D["merged vocabulary"]
    D --> B
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```


In [ ]:
# -- BPE helpers: initialize, count, and merge -----------------------------------
def initialize_vocabulary(corpus):
    words = Counter(re.findall(r"[a-z-]+", " ".join(corpus).lower()))
    return {tuple(word) + ("</w>",): count for word, count in words.items()}

def count_pairs(vocabulary):
    pairs = Counter()
    for symbols, frequency in vocabulary.items():
        for pair in zip(symbols, symbols[1:]):
            pairs[pair] += frequency
    return pairs

def merge_best_pair(vocabulary, pair):
    updated = {}
    for symbols, frequency in vocabulary.items():
        merged = []
        index = 0
        while index < len(symbols):
            if index + 1 < len(symbols) and (symbols[index], symbols[index + 1]) == pair:
                merged.append(symbols[index] + symbols[index + 1])
                index += 2
            else:
                merged.append(symbols[index])
                index += 1
        updated[tuple(merged)] = frequency
    return updated

vocabulary = initialize_vocabulary(LEGAL_CORPUS)
learned_merges = []
for _ in range(20):
    pairs = count_pairs(vocabulary)
    if not pairs:
        break
    pair = pairs.most_common(1)[0][0]
    learned_merges.append(pair)
    vocabulary = merge_best_pair(vocabulary, pair)

print(f"Learned merges: {len(learned_merges)}")
print("First five:", learned_merges[:5])


In [ ]:
# -- Apply learned merges to a new word -------------------------------------------
def apply_merges(word, merges):
    symbols = list(word) + ["</w>"]
    for pair in merges:
        output = []
        index = 0
        while index < len(symbols):
            if index + 1 < len(symbols) and (symbols[index], symbols[index + 1]) == pair:
                output.append(symbols[index] + symbols[index + 1])
                index += 2
            else:
                output.append(symbols[index])
                index += 1
        symbols = output
    return symbols

phrase_word = "non-disclosure"
pieces = apply_merges(phrase_word, learned_merges)
print(f"{phrase_word!r} -> {pieces}")
print(f"Compression: {len(phrase_word) + 1} character symbols -> {len(pieces)} BPE pieces")


### Your turn - change the merge budget

**Your turn:** change only the number of learned merges and predict the target piece count.


In [ ]:
# -- Your turn: one-variable BPE budget experiment -------------------------------
# CHANGE THIS: try 5 or 12
merge_budget = 8
tested_pieces = apply_merges("non-disclosure", learned_merges[:merge_budget])
print(f"Budget {merge_budget}: {tested_pieces} ({len(tested_pieces)} pieces)")
print("Check: using more already-learned merges cannot increase this word's piece count.")


## 2 - Inspect the production GPT-2 tokenizer

The toy tokenizer uses characters. GPT-2 starts from bytes and applies a pretrained merge table.

**Predict:** will `" agreement"` and `"agreement"` necessarily receive the same IDs? GPT-2 tokens can include leading whitespace.


In [ ]:
# -- GPT-2 tiktoken: IDs, bytes, whitespace, and round-trip -----------------------
encoder = tiktoken.get_encoding("gpt2")
phrase = "The non-disclosure agreement protects data."
token_ids = encoder.encode(phrase)
token_bytes = [encoder.decode_single_token_bytes(token_id) for token_id in token_ids]
decoded = encoder.decode(token_ids)

print(f"Text: {phrase!r}")
print(f"Token IDs: {token_ids}")
print(f"Token bytes: {token_bytes}")
print(f"Round-trip exact: {decoded == phrase}")
print(f"' agreement' IDs: {encoder.encode(' agreement')}")
print(f"'agreement' IDs:  {encoder.encode('agreement')}")
assert decoded == phrase


In [ ]:
# -- Compression evidence on the legal corpus ------------------------------------
legal_text = "\n".join(LEGAL_CORPUS)
production_ids = encoder.encode(legal_text)
characters_per_token = len(legal_text) / len(production_ids)
print(f"Characters: {len(legal_text)}")
print(f"GPT-2 tokens: {len(production_ids)}")
print(f"Characters per token: {characters_per_token:.2f}")
print("Boundary: this measures this corpus, not every language or legal domain.")


## 3 - Train embedding rows under controlled prediction pressure

```mermaid
flowchart LR
    A["input token ID"] --> B["nn.Embedding row"]
    B --> C["linear next-token head"]
    C --> D["cross-entropy"]
    D --> E["gradient updates selected rows"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```


In [ ]:
# -- Controlled corpus and nn.Embedding model ------------------------------------
lessons = [
    ("contract", "binds"),
    ("agreement", "binds"),
    ("contract", "protects"),
    ("agreement", "protects"),
    ("invoice", "lists"),
    ("invoice", "amount"),
]
words = sorted({word for pair in lessons for word in pair})
word_to_id = {word: index for index, word in enumerate(words)}
input_ids = torch.tensor([word_to_id[source] for source, _ in lessons], dtype=torch.long)
target_ids = torch.tensor([word_to_id[target] for _, target in lessons], dtype=torch.long)

class LookupPredictor(nn.Module):
    def __init__(self, vocabulary_size, dimension=4):
        super().__init__()
        self.embedding = nn.Embedding(vocabulary_size, dimension)
        self.head = nn.Linear(dimension, vocabulary_size, bias=False)

    def forward(self, ids):
        return self.head(self.embedding(ids))

model = LookupPredictor(len(words))
before = model.embedding.weight.detach().clone()
optimizer = torch.optim.Adam(model.parameters(), lr=0.08)
losses = []

for _ in range(250):
    logits = model(input_ids)
    loss = nn.functional.cross_entropy(logits, target_ids)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    losses.append(float(loss.detach()))

after = model.embedding.weight.detach().clone()
contract_id = word_to_id["contract"]
agreement_id = word_to_id["agreement"]
before_distance = torch.dist(before[contract_id], before[agreement_id]).item()
after_distance = torch.dist(after[contract_id], after[agreement_id]).item()

print(f"Loss: {losses[0]:.4f} -> {losses[-1]:.4f}")
print(f"contract/agreement distance: {before_distance:.4f} -> {after_distance:.4f}")
print("Takeaway: shared prediction pressure changed the addressed rows.")
assert losses[-1] < losses[0]


In [ ]:
# -- Only participating rows receive embedding gradients in one lesson ----------
probe_model = LookupPredictor(len(words))
probe_logits = probe_model(torch.tensor([word_to_id["contract"]]))
probe_loss = nn.functional.cross_entropy(
    probe_logits,
    torch.tensor([word_to_id["binds"]]),
)
probe_loss.backward()
row_gradient_norms = probe_model.embedding.weight.grad.norm(dim=1)
nonzero_rows = [words[index] for index in torch.nonzero(row_gradient_norms).flatten().tolist()]
print(f"Embedding rows with nonzero gradient: {nonzero_rows}")
print("Takeaway: lookup makes sparse row participation visible.")
assert nonzero_rows == ["contract"]


## 4 - Pad the batch, then prove padding contributes zero loss

```mermaid
flowchart LR
    A["variable-length IDs"] --> B["padded rectangle"]
    B --> C["Boolean real-token mask"]
    B --> W["wrong mean over all cells"]
    C --> R["explicit real-token reduction"]
    C --> I["ignore_index reduction"]
    R --> M["matching correct loss"]
    I --> M
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style W fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style R fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style I fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style M fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```


In [ ]:
# -- Padded batch and mask visualization -----------------------------------------
PAD = "<pad>"
batch_tokens = [
    ["the", "contract", "binds"],
    ["the", "agreement"],
    ["invoice"],
]
batch_vocab = [PAD] + sorted({token for row in batch_tokens for token in row})
batch_to_id = {token: index for index, token in enumerate(batch_vocab)}
pad_id = batch_to_id[PAD]
max_length = max(map(len, batch_tokens))
padded = torch.full((len(batch_tokens), max_length), pad_id, dtype=torch.long)
for row_index, row in enumerate(batch_tokens):
    padded[row_index, : len(row)] = torch.tensor([batch_to_id[token] for token in row])
mask = padded.ne(pad_id)

fig, ax = plt.subplots(figsize=(8, 3))
image = ax.imshow(mask.numpy(), cmap="Blues", vmin=0, vmax=1)
ax.set(xticks=range(max_length), yticks=range(len(batch_tokens)))
ax.set(xlabel="Position", ylabel="Sequence", title="PyTorch real-token mask")
plt.colorbar(image, ax=ax)
plt.show()
print(padded)
print(mask)


In [ ]:
# -- Wrong versus correct PyTorch reductions -------------------------------------
torch.manual_seed(7)
vocabulary_size = len(batch_vocab)
logits = torch.randn(*padded.shape, vocabulary_size)
targets = padded.clone()
per_position = nn.functional.cross_entropy(
    logits.reshape(-1, vocabulary_size),
    targets.reshape(-1),
    reduction="none",
)
flat_targets = targets.reshape(-1)
real = flat_targets.ne(pad_id)

wrong_loss = per_position.mean()
explicit_loss = per_position[real].sum() / real.sum()
ignore_index_loss = nn.functional.cross_entropy(
    logits.reshape(-1, vocabulary_size),
    flat_targets,
    ignore_index=pad_id,
)

print(f"Wrong all-cell mean: {wrong_loss.item():.6f}")
print(f"Explicit masked mean: {explicit_loss.item():.6f}")
print(f"ignore_index mean:    {ignore_index_loss.item():.6f}")
print(f"Correct methods match: {torch.allclose(explicit_loss, ignore_index_loss)}")
assert torch.allclose(explicit_loss, ignore_index_loss)


In [ ]:
# -- Padding-row contract: padding_idx prevents row updates ----------------------
embedding = nn.Embedding(len(batch_vocab), 5, padding_idx=pad_id)
vectors = embedding(padded)
vectors.sum().backward()
pad_gradient = embedding.weight.grad[pad_id].abs().max().item()
real_gradient = embedding.weight.grad[batch_to_id["contract"]].norm().item()
print(f"Maximum pad-row gradient: {pad_gradient:.6f}")
print(f"Contract-row gradient norm: {real_gradient:.6f}")
assert pad_gradient == 0.0
assert real_gradient > 0.0


### Your turn - add padding without changing the answer

**Your turn:** append one extra pad column to both logits and targets. The correct loss should remain unchanged; the wrong mean usually will not.


In [ ]:
# -- Your turn: padding invariance of the correct objective -----------------------
# CHANGE THIS: try 1 or 3 extra pad columns
extra_pad_columns = 2
extra_logits = torch.randn(len(batch_tokens), extra_pad_columns, vocabulary_size)
extended_logits = torch.cat([logits, extra_logits], dim=1)
extended_targets = torch.cat(
    [targets, torch.full((len(batch_tokens), extra_pad_columns), pad_id, dtype=torch.long)],
    dim=1,
)
extended_loss = nn.functional.cross_entropy(
    extended_logits.reshape(-1, vocabulary_size),
    extended_targets.reshape(-1),
    ignore_index=pad_id,
)
print(f"Original correct loss: {ignore_index_loss.item():.6f}")
print(f"Extended correct loss: {extended_loss.item():.6f}")
print(f"Padding invariant: {torch.allclose(ignore_index_loss, extended_loss)}")
assert torch.allclose(ignore_index_loss, extended_loss)


## 5 - Evidence scorecard and Transformer handoff

| Contract | Measured evidence |
|---|---|
| BPE merge budget | target piece count printed from learned merges |
| production tokenizer | GPT-2 IDs, bytes, whitespace, and round-trip inspected |
| IDs are addresses | `nn.Embedding` row participation measured |
| rows learn from use | controlled loss and row-distance change |
| padding is not evidence | explicit reduction equals `ignore_index` |
| padding row stays inert | `padding_idx` row gradient is zero |

**Built and measured:** BPE, GPT-2 inspection, trainable lookup, padded batch, loss masking.

**Explained or illustrated:** static versus contextual boundary.

**Named and out of scope:** Word2Vec/GloVe history (optional branch), semantic search (RAG), and causal attention masking (Transformers).

You now enter Transformers with token IDs, embedding rows, batch masks, and loss masks already separated. The next new problem is contextual routing, not basic text preprocessing.
